In [ ]:
import SimpleITK as sitk
import numpy as np
import os

def get_mask_bounds(mask_image):
    """
    ?????????????

    ???
    - mask_image: SimpleITK ??????????

    ???
    - bounds: ???? (xmin, xmax, ymin, ymax, zmin, zmax)
    """
    # ?????? NumPy ??
    mask_array = sitk.GetArrayFromImage(mask_image)
    # ???????????
    non_zero_indices = np.argwhere(mask_array)

    if non_zero_indices.size == 0:
        # ????
        return None

    # ??????????????
    z_indices, y_indices, x_indices = non_zero_indices[:,0], non_zero_indices[:,1], non_zero_indices[:,2]
    xmin, xmax = x_indices.min(), x_indices.max()
    ymin, ymax = y_indices.min(), y_indices.max()
    zmin, zmax = z_indices.min(), z_indices.max()

    return (xmin, xmax, ymin, ymax, zmin, zmax)

def calculate_rectum_bounding_box(hip_right_path, hip_left_path, prostate_path, sacrum_path, image_size):
    """
    ??????????????????

    ???
    - hip_right_path: hip_right ????????
    - hip_left_path: hip_left ????????
    - prostate_path: prostate ????????
    - sacrum_path: sacrum ????????
    - image_size: ??????? (size_x, size_y, size_z)

    ???
    - rectum_bbox: ???? (xmin, xmax, ymin, ymax, zmin, zmax)
    """
    # ????
    hip_right_mask = sitk.ReadImage(hip_right_path)
    hip_left_mask = sitk.ReadImage(hip_left_path)
    prostate_mask = sitk.ReadImage(prostate_path)
    sacrum_mask = sitk.ReadImage(sacrum_path)

    # ?????????
    hip_right_bounds = get_mask_bounds(hip_right_mask)
    hip_left_bounds = get_mask_bounds(hip_left_mask)
    prostate_bounds = get_mask_bounds(prostate_mask)
    sacrum_bounds = get_mask_bounds(sacrum_mask)

    # ?????????
    if any(b is None for b in [hip_right_bounds, hip_left_bounds, prostate_bounds, sacrum_bounds]):
        raise ValueError("??????????")

    # ?? Xmin ? Xmax
    Xmin = hip_right_bounds[1]  # hip_right ? xmax
    Xmax = hip_left_bounds[0]   # hip_left ? xmin

    # ?? Ymax ? Ymin
    Ymax = prostate_bounds[2]   # prostate ? ymin
    Ymin = sacrum_bounds[3]     # sacrum ? ymax

    # Zmin ? Zmax ??????????
    Zmin = 0
    Zmax = image_size[2] - 1

    rectum_bbox = (Xmin, Xmax, Ymin, Ymax, Zmin, Zmax)
    return rectum_bbox

def extract_patch_from_image(image_path, rectum_bbox, output_path):
    """
    ??????????????????

    ???
    - image_path: ????????
    - rectum_bbox: ????? (xmin, xmax, ymin, ymax, zmin, zmax)
    - output_path: ???????????

    ???
    - None
    """
    # ??????
    image = sitk.ReadImage(image_path)

    # ??????
    extract_size = [int(rectum_bbox[1] - rectum_bbox[0] + 1),
                    int(rectum_bbox[3] - rectum_bbox[2] + 1),
                    int(rectum_bbox[5] - rectum_bbox[4] + 1)]

    extract_index = [int(rectum_bbox[0]),
                     int(rectum_bbox[2]),
                     int(rectum_bbox[4])]

    # ??????????????
    image_size = image.GetSize()
    for i in range(3):
        if extract_index[i] < 0 or extract_index[i] + extract_size[i] > image_size[i]:
            raise ValueError("????????????")

    # ????
    extract_filter = sitk.ExtractImageFilter()
    extract_filter.SetSize(extract_size)
    extract_filter.SetIndex(extract_index)
    patch = extract_filter.Execute(image)

    # ???????
    sitk.WriteImage(patch, output_path)

    print(f"?????????: {output_path}")

# ?????

# ???????????????????
path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/train/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex'
hip_right = 'hip_right.nii.gz'
hip_left = 'hip_left.nii.gz'
prostate= 'prostate.nii.gz'
sacrum = 'sacrum.nii.gz'

hip_right_path = os.path.join(path, hip_right)
hip_left_path = os.path.join(path, hip_left)
prostate_path= os.path.join(path, prostate)
sacrum_path = os.path.join(path, sacrum)


# ???????
original_image_path = r'/path/to/workspace/classification_model_originalimage/train_data/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex.nii.gz'

# ???????????
path2 = r'/path/to/workspace/classification_model_originalimage'
output_patch = 'extracted_patch2.nii.gz'
output_patch_path = os.path.join(path2, output_patch)

# ????????????
original_image = sitk.ReadImage(original_image_path)
image_size = original_image.GetSize()

# ????????
rectum_bbox = calculate_rectum_bounding_box(hip_right_path, hip_left_path, prostate_path, sacrum_path, image_size)

print("??????? (xmin, xmax, ymin, ymax, zmin, zmax):")
print(rectum_bbox)

# ???????
extract_patch_from_image(original_image_path, rectum_bbox, output_patch_path)
